<a href="https://colab.research.google.com/github/sipy/su5-phase-field/blob/main/4_Matter_Antimatter_Annihilation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn.functional as F
import math
import time

# ==========================================
# 1. THE FUSION COLLIDER HARDWARE SETUP
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float64)
print(f"Hardware acquired: {device.type.upper()} (FLOAT64 / FUSION MODE)")

N = 128
dx = 0.08

mask = torch.zeros(1, 4, N, N, N, device=device)
mask[:, :, 2:-2, 2:-2, 2:-2] = 1.0
vacuum = torch.tensor([1.0, 0.0, 0.0, 0.0], device=device).view(1, 4, 1, 1, 1)

# ==========================================
# 2. THE TOPOLOGICAL SEED GENERATOR
# ==========================================
X, Y, Z = torch.meshgrid(torch.arange(N, device=device),
                         torch.arange(N, device=device),
                         torch.arange(N, device=device), indexing='ij')

def create_seed(cx, cy, cz, B_charge):
    """Generates a topological knot of charge B at a specific coordinate"""
    rx = (X - cx) * dx
    ry = (Y - cy) * dx
    rz = (Z - cz) * dx
    r = torch.sqrt(rx**2 + ry**2 + rz**2) + 1e-8

    # Convert to spherical coordinates for the internal twist
    theta = torch.acos(rz / r)
    phi_angle = torch.atan2(ry, rx)

    R_knot = 1.0
    f = math.pi * torch.exp(-r / R_knot)

    # The B_charge multiplies the azimuthal angle to wrap the sphere B times!
    phi0 = torch.cos(f)
    phi1 = torch.sin(f) * torch.sin(theta) * torch.cos(B_charge * phi_angle)
    phi2 = torch.sin(f) * torch.sin(theta) * torch.sin(B_charge * phi_angle)
    phi3 = torch.sin(f) * torch.cos(theta)

    return torch.stack([phi0, phi1, phi2, phi3], dim=0)

print("Loading the Reactor: Generating Deuterium (B=2) and Tritium (B=3)...")

# Place B=2 on the Left (offset by -15 pixels)
seed_D = create_seed(N//2 - 15, N//2, N//2, B_charge=1)

# Place B=3 on the Right (offset by +15 pixels)
seed_T = create_seed(N//2 + 15, N//2, N//2, B_charge=-1)

# ==========================================
# 3. THE STEREOGRAPHIC FUSION MULTIPLICATION
# ==========================================
# We cannot add them. We must multiply their SU(2) representations.
# U_tot = U_1 * U_2
# Real part:  phi0_1 * phi0_2 - dot_product(vec_1, vec_2)
# Imag part:  phi0_1 * vec_2 + phi0_2 * vec_1 - cross_product(vec_1, vec_2)

p0_D, vec_D = seed_D[0], seed_D[1:]
p0_T, vec_T = seed_T[0], seed_T[1:]

# Dot product of the vector parts
dot_DT = torch.sum(vec_D * vec_T, dim=0)

# Cross product of the vector parts
cross_DT = torch.cross(vec_D, vec_T, dim=0)

# Combine them into the unified Fusion Field
phi0_tot = p0_D * p0_T - dot_DT
vec_tot = p0_D * vec_T + p0_T * vec_D - cross_DT

phi = torch.cat([phi0_tot.unsqueeze(0), vec_tot], dim=0).unsqueeze(0)
phi = F.normalize(phi, p=2, dim=1)
phi = phi * mask + vacuum * (1.0 - mask)
phi.requires_grad_(True)

# ==========================================
# 4. TUNED 4TH-ORDER PHYSICS ENGINE
# ==========================================
def get_derivatives_4th_order(phi_tensor, dx_val):
    k_vals = [1.0/12.0, -8.0/12.0, 0.0, 8.0/12.0, -1.0/12.0]
    kernel = torch.tensor(k_vals, device=device) / dx_val
    kx = kernel.view(1, 1, 1, 1, 5).repeat(4, 1, 1, 1, 1)
    ky = kernel.view(1, 1, 1, 5, 1).repeat(4, 1, 1, 1, 1)
    kz = kernel.view(1, 1, 5, 1, 1).repeat(4, 1, 1, 1, 1)

    pad_x = F.pad(phi_tensor, (2, 2, 0, 0, 0, 0), mode='replicate')
    dx_phi = F.conv3d(pad_x, kx, groups=4)
    pad_y = F.pad(phi_tensor, (0, 0, 2, 2, 0, 0), mode='replicate')
    dy_phi = F.conv3d(pad_y, ky, groups=4)
    pad_z = F.pad(phi_tensor, (0, 0, 0, 0, 2, 2), mode='replicate')
    dz_phi = F.conv3d(pad_z, kz, groups=4)

    return torch.stack([dx_phi, dy_phi, dz_phi], dim=0)

def compute_skyrme_energy(phi_tensor, dx_val, f_pi=1.0, e=1.0, m_pi=0.15):
    dphi = get_derivatives_4th_order(phi_tensor, dx_val)
    G = torch.einsum('iczyx, jczyx -> ijzyx', dphi[:, 0], dphi[:, 0])

    E2 = G[0,0] + G[1,1] + G[2,2]
    TrG_sq = E2**2
    Tr_G2 = sum(G[i,j] * G[j,i] for i in range(3) for j in range(3))
    E4 = 0.5 * (TrG_sq - Tr_G2)
    E_mass = (m_pi**2) * (1.0 - phi_tensor[0, 0])

    energy_density = (f_pi**2 / 16.0) * E2 + (1.0 / (32.0 * e**2)) * E4 + E_mass
    return torch.sum(energy_density) * (dx_val**3)

def compute_winding_number(phi_tensor, dx_val):
    dphi = get_derivatives_4th_order(phi_tensor, dx_val)
    vectors = torch.stack([phi_tensor[0], dphi[0,0], dphi[1,0], dphi[2,0]], dim=0)
    matrix_field = vectors.permute(2, 3, 4, 0, 1)
    det_density = torch.linalg.det(matrix_field)
    return torch.sum(det_density) * (dx_val**3) / (2 * math.pi**2)

# ==========================================
# 5. THE COLLISION SEQUENCE
# ==========================================
with torch.no_grad():
    initial_B = compute_winding_number(phi, dx)
    print(f"Pre-Collision System Winding B: {initial_B.item():.5f}\n")

optimizer = torch.optim.Adam([phi], lr=0.015)
print("Igniting Fusion Relaxation...")
start_time = time.time()

for step in range(1501):
    optimizer.zero_grad()

    energy = compute_skyrme_energy(phi, dx)
    energy.backward()

    phi.grad.clamp_(-0.02, 0.02)
    optimizer.step()

    with torch.no_grad():
        phi.data = phi.data * mask + vacuum * (1.0 - mask)
        phi.copy_(F.normalize(phi, p=2, dim=1))

    if step % 100 == 0:
        with torch.no_grad():
            B = compute_winding_number(phi, dx)
            elapsed = time.time() - start_time
            print(f"Step {step:4d} | Time: {elapsed:.1f}s | Energy: {energy.item():.4f} | System Winding B: {B.item():.5f}")

print("\nCollision Complete. The nuclei have merged.")

Hardware acquired: CUDA (FLOAT64 / FUSION MODE)
Loading the Reactor: Generating Deuterium (B=2) and Tritium (B=3)...
Pre-Collision System Winding B: -0.00000

Igniting Fusion Relaxation...
Step    0 | Time: 0.5s | Energy: 14.6149 | System Winding B: -0.00000
Step  100 | Time: 27.4s | Energy: 14.3151 | System Winding B: 0.00001
Step  200 | Time: 54.7s | Energy: 14.4230 | System Winding B: 0.00000
Step  300 | Time: 82.4s | Energy: 14.4934 | System Winding B: 0.00000
Step  400 | Time: 110.8s | Energy: 14.5436 | System Winding B: -0.00000
Step  500 | Time: 139.5s | Energy: 14.5844 | System Winding B: -0.00000
Step  600 | Time: 167.9s | Energy: 14.6163 | System Winding B: 0.00000
Step  700 | Time: 196.3s | Energy: 14.6441 | System Winding B: -0.00000
Step  800 | Time: 224.8s | Energy: 14.6700 | System Winding B: -0.00000
Step  900 | Time: 253.2s | Energy: 14.6920 | System Winding B: -0.00000
Step 1000 | Time: 281.6s | Energy: 14.7137 | System Winding B: -0.00001
Step 1100 | Time: 310.1s | E

In [4]:
import torch
import numpy as np
import plotly.graph_objects as go
from skimage import measure

# Check if the simulation was actually run in this session
if 'phi' not in globals():
    print("CRITICAL ERROR: 'phi' not found in memory.")
    print("Please run your simulation cell (Step 2) before running this visualizer.")
else:
    print("Capturing the Aftermath: Extracting Radiation Field from GPU...")

    with torch.no_grad():
        # (1.0 - phi_0) highlights the topological 'energy' left behind
        volume = (1.0 - phi[0, 0, :, :, :]).cpu().numpy()

    try:
        # We search for the 'scars' in the field.
        # A level of 0.1 is usually perfect for catching the ripples.
        verts, faces, normals, values = measure.marching_cubes(volume, level=0.1)

        print("Rendering Annihilation Isosurface...")

        fig = go.Figure(data=[go.Mesh3d(
            x=verts[:, 0],
            y=verts[:, 1],
            z=verts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            opacity=0.5,
            color='indianred',
            flatshading=True
        )])

        fig.update_layout(
            title='Matter-Antimatter Annihilation (Radiation Field Aftermath)',
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                bgcolor='rgb(10, 10, 10)'
            ),
            margin=dict(l=0, r=0, b=0, t=40)
        )

        fig.show()

    except ValueError:
        print("Zero-Point Warning: The annihilation was so complete that no energy ")
        print("ripples remain above the 0.1 threshold. The knots are fully untied.")

Capturing the Aftermath: Extracting Radiation Field from GPU...
Rendering Annihilation Isosurface...
